In [ ]:
# 音声処理とAIモデルに必要なライブラリをインストール
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q google-genai pydub

メニューから [ランタイム] ＞ [ランタイムのタイプを変更] を開き、ハードウェアアクセラレータを [T4 GPU]（または利用可能なGPU）に変更して保存します。文字起こしの速度が劇的に上がります。

In [ ]:
import os
import whisper
from google import genai
from google.colab import userdata

# 1. 音声ファイルの設定 (Colabにアップロードしたファイル名に変えてください)
# ※左のフォルダアイコンからドラッグ＆ドロップでアップロードできます
AUDIO_FILE = "meeting_audio.mp3"

# 2. Gemini APIの準備
# Colabの「シークレット(鍵アイコン)」に GEMINI_API_KEY を登録しておいてください
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
except Exception as e:
    print("APIキーが設定されていないか、エラーが発生しました。シークレットを確認してください。")
    client = None

def create_minutes(audio_path):
    if not os.path.exists(audio_path):
        print(f"エラー: 指定された音声ファイル '{audio_path}' が見つかりません。")
        return

    # --- Step 1: Whisperによる文字起こし ---
    print("🤖 1/2: 音声の文字起こしを開始します... (数分かかる場合があります)")
    # モデルは base / small / medium / large-v3 などから選べます（日本語はsmall以上がおすすめ）
    model = whisper.load_model("small")
    result = model.transcribe(audio_path, verbose=False, language="ja")
    transcription = result["text"]

    print("\n--- 文字起こし結果（冒頭部分） ---")
    print(transcription[:300] + "...\n")

    # --- Step 2: Geminiによる要約と議事録化 ---
    if client is None:
        print("Gemini APIキーがないため、文字起こしのみで終了します。")
        return

    print("🧠 2/2: Geminiで議事録を生成しています...")

    prompt = f"""
以下の会議の文字起こしデータから、綺麗なビジネス議事録を作成してください。

# 成果物の構成案：
1. 会議のテーマ・目的（文字起こしから推測）
2. 決定事項
3. 次回への宿題・タスク（担当者がわかれば記載）
4. 会議の要約・詳細（時系列またはトピックごとに見出しをつけて整理）

# 文字起こしデータ：
{transcription}
"""

    # 最新の推奨モデル「gemini-2.5-flash」を使用
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
    )

    minutes_text = response.text

    # --- Step 3: 結果の保存 ---
    output_file = "minutes.md"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(minutes_text)

    print(f"✨ 議事録の生成が完了しました！ '{output_file}' に保存されました。")
    print("\n--- 生成された議事録 ---")
    print(minutes_text)

# 実行
create_minutes(AUDIO_FILE)

一時間以上の議事録の場合

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# AUDIO_FILE = "/content/drive/MyDrive/meeting.mp3" のように指定